# 🚀 Google AI Ultra (5x / 20x) 云端 AI 工作站 (MCP Agent 托管版)
- **连接方式**：`colab-mcp` 网页桥接 / `colab` CLI + `colab-dfs-ext` 无头零浏览器直连
- **存储底座**：20TB/30TB Google Drive 云端内网直连 (`/content/drive/MyDrive/AI_Workstation`)

### 🧠 一、AI 视频与多模态任务硬件调度表
| 任务类型 (`task_type`) | 推荐运行时 | 显存 / 架构 | 费率 (CCU/h) | Agent 自动调优与执行策略 |
| :--- | :--- | :--- | :--- | :--- |
| **`download` / `preprocess`** (下载模型/处理文件) | **`CPU`** | 0 GB | **0 CCU/h** | **严禁开高价 GPU 下载大模型**；若处于 G4/A100/H100，Agent 必须先拦截提醒 |
| **`light_video`** (Wan2.1-1.3B / LTX / Flux / 调试) | **`L4 GPU`** | 22.5 GB (Ada) | **1.71 CCU/h** | 开启 FP8/BF16 + `enable_model_cpu_offload()` |
| **`heavy_video`** (Wan2.1-14B / Hunyuan / 1080p 渲染) | **`G4 GPU`** (不勾高RAM) | **96 GB (Blackwell)** | **6.7~8.71 CCU/h** | **主力王炸卡**；关闭 `cpu_offload`（100% 驻留 96G 显存），开启原生 FP8/BF16 + `sdpa` |

### 🤖 二、基础大语言模型 (LLM) 尺寸与硬件匹配表（推理 & 微调）
| LLM 参数规模 (代表模型) | 4-bit 量化推理 (AWQ/GGUF/FP4) | 原生 BF16 / FP8 满血推理 | LoRA / QLoRA 微调 (Unsloth) | 核心选卡逻辑 (显存与架构第一性原理) |
| :--- | :--- | :--- | :--- | :--- |
| **1.5B ~ 9B** (`llm_small`)<br>*Qwen3-8B / Llama-3.1-8B / Gemma-9B* | **`T4 GPU`** (15GB)<br>*1.19 CCU/h (占~6G)* | **`L4 GPU`** (22.5GB)<br>*1.71 CCU/h (BF16占~16G, FP8占~9G)* | **`L4 GPU`** (22.5GB)<br>*1.71 CCU/h* | 8B 级别用 **L4** 即可满血跑 BF16/FP8 及微调，千万别开 A100/G4 浪费算力 |
| **14B ~ 32B** (`llm_medium`)<br>*Qwen-14B/32B / DeepSeek-R1-32B* | **`L4 GPU`** (22.5GB)<br>*1.71 CCU/h (32B Q4占~19G 极限卡位)* | **14B (FP8)** 👉 **`L4 GPU`**<br>**32B (BF16/FP8)** 👉 **`G4 GPU`** (96GB) | **14B QLoRA** 👉 **`L4 GPU`**<br>**32B 微调** 👉 **`G4 GPU`** (96GB) | **32B 是分水岭**：32B Q4 刚好塞进 L4 (最省钱)；32B 满血 BF16 (~65G) 40G A100 装不下，必须上 **G4 (96GB)** |
| **70B ~ 72B** (`llm_large`)<br>*Llama-3.3-70B / Qwen-72B* | **`G4 GPU`** (96GB)<br>*8.71 CCU/h (独占原生 NVFP4 极速推理)* | **`G4 GPU`** (96GB)<br>*8.71 CCU/h (70B FP8占~72G + 24G 长上下文)* | **`G4 GPU`** (96GB)<br>*8.71 CCU/h (70B QLoRA占~48G)* | **70B 选 G4 胜过 80G A100/H100**：70B FP8 权重占 72G，80G 卡只剩 8G 给上下文易 OOM；**G4 96G 可留 24G 给超长上下文**！ |

In [1]:
# === 1. 智能硬件识别、费率监控与任务匹配顾问 ( 视频 + 全尺寸 LLM ) ===
import platform, shutil, torch

GPU_PROFILES = {
    "CPU":          {"tier": "CPU",  "ccu_h": 0.0,   "arch": "Host CPU",          "fp8": False, "fp4": False},
    "T4":           {"tier": "T4",   "ccu_h": 1.19,  "arch": "Turing (sm_75)",    "fp8": False, "fp4": False},
    "L4":           {"tier": "L4",   "ccu_h": 1.71,  "arch": "Ada (sm_89)",       "fp8": True,  "fp4": False},
    "A100":         {"tier": "A100", "ccu_h": 5.40,  "arch": "Ampere (sm_80)",    "fp8": False, "fp4": False},
    "RTX PRO 6000": {"tier": "G4",   "ccu_h": 8.71,  "arch": "Blackwell (sm_120)","fp8": True,  "fp4": True},
    "Blackwell":    {"tier": "G4",   "ccu_h": 8.71,  "arch": "Blackwell (sm_120)","fp8": True,  "fp4": True},
    "H100":         {"tier": "H100", "ccu_h": 18.05, "arch": "Hopper (sm_90)",    "fp8": True,  "fp4": False},
}

TASK_RECOMMENDATIONS = {
    "download":    {"best": ["CPU"],        "desc": "下载模型 / 整理云盘 (0 CCU)"},
    "light_video": {"best": ["L4", "T4"],   "desc": "轻量视频/图像生成 (<20GB 显存, Wan2.1-1.3B / Flux)"},
    "heavy_video": {"best": ["G4"],         "desc": "14B 满血视频生成 (Wan2.1-14B / Hunyuan, 需 96GB G4 全驻留)"},
    "llm_small":   {"best": ["L4", "T4"],   "desc": "1.5B~9B LLM (Q4 选 T4; BF16/FP8/微调 选 L4)"},
    "llm_medium":  {"best": ["L4", "G4"],   "desc": "14B~32B LLM (14B FP8 / 32B Q4 选 L4; 32B BF16/微调 选 G4 96GB)"},
    "llm_large":   {"best": ["G4", "H100"], "desc": "70B~72B LLM (FP8 长上下文 / NVFP4 / QLoRA 首选 G4 96GB)"},
}

def inspect_hardware(task_type: str = "general"):
    """
    task_type 可选:
      - 'download'    : 下载模型 / 挂载云盘 (推荐 CPU)
      - 'light_video' : 轻量视频/生图 (<20GB 显存，推荐 L4)
      - 'heavy_video' : 14B 满血视频大模型 (推荐 G4 96GB Blackwell)
      - 'llm_small'   : 1.5B ~ 9B LLM 推理或微调 (推荐 L4 / T4)
      - 'llm_medium'  : 14B ~ 32B LLM 推理或微调 (32B Q4 选 L4; 32B 满血/微调 选 G4)
      - 'llm_large'   : 70B ~ 72B LLM 推理或微调 (推荐 G4 96GB 跑 FP8/NVFP4/QLoRA)
      - 'general'     : 仅查看当前状态
    """
    if not torch.cuda.is_available():
        gpu_name, vram_gb, sm = "CPU", 0.0, (0, 0)
        profile = GPU_PROFILES["CPU"]
    else:
        gpu_name = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        vram_gb = props.total_memory / (1024**3)
        sm = (props.major, props.minor)
        profile = next((v for k, v in GPU_PROFILES.items() if k != "CPU" and k.lower() in gpu_name.lower()),
                       {"tier": "GPU", "ccu_h": 5.0, "arch": f"sm_{sm[0]}{sm[1]}", "fp8": sm[0] >= 9 or sm == (8, 9), "fp4": sm[0] >= 10})
        if profile["tier"] == "A100" and vram_gb > 60:
            profile = {**profile, "tier": "A100-80G", "ccu_h": 7.52}

    ccu_h = profile["ccu_h"]
    hours_left = f"{1000 / ccu_h:.0f} 小时" if ccu_h > 0 else "♾️ 无限 (0 CCU)"

    hw_config = {
        "tier": profile["tier"],
        "gpu_name": gpu_name,
        "vram_gb": round(vram_gb, 1),
        "ccu_per_hour": ccu_h,
        "native_fp8": profile["fp8"],
        "native_fp4": profile["fp4"],
        "enable_cpu_offload": vram_gb < 45.0,
        "recommended_dtype": "fp8_e4m3fn / bfloat16" if profile["fp8"] else "bfloat16 / float16",
        "attn_backend": "sdpa",
    }

    print(f"🖥️ 当前运行时: [{hw_config['tier']}] {gpu_name} | 显存: {vram_gb:.1f} GB | 架构: {profile['arch']}")
    print(f"💰 算力费率  : {ccu_h} CCU/小时 (月度续航: {hours_left})")
    print(f"⚙️ 自动调优  : Offload={hw_config['enable_cpu_offload']} | 推荐精度={hw_config['recommended_dtype']} | 原生FP8={profile['fp8']} | 原生FP4={profile['fp4']}")

    if task_type in TASK_RECOMMENDATIONS:
        rec = TASK_RECOMMENDATIONS[task_type]
        best_str = " / ".join(rec["best"])
        if hw_config["tier"] in rec["best"]:
            print(f"\n✅ [硬件匹配] 当前 [{hw_config['tier']}] 完美匹配任务「{rec['desc']}」！")
        elif task_type == "download" and ccu_h >= 5.0:
            print(f"\n🚨 [防烧钱警告] 当前正挂载高价显卡 [{hw_config['tier']}] ({ccu_h} CCU/h)！首次下载几十 GB 模型建议切回 CPU (0 CCU)！")
        elif ccu_h > 2.0 and task_type in ("llm_small", "light_video"):
            print(f"\n💸 [大炮打蚊子预警] 任务「{rec['desc']}」仅需 [{best_str}] (1.19~1.71 CCU/h)，当前 [{hw_config['tier']}] ({ccu_h} CCU/h) 偏贵！")
        else:
            print(f"\n💡 [选卡建议] 任务「{rec['desc']}」最优推荐为 [{best_str}]，当前为 [{hw_config['tier']}]。")
    return hw_config

HW_CONFIG = inspect_hardware("general")

🖥️ 当前运行时: [CPU] CPU | 显存: 0.0 GB | 架构: Host CPU
💰 算力费率  : 0.0 CCU/小时 (按 1,000 CCU 测算可跑: ♾️ 无限 (0 CCU))
⚙️ 自动调优  : Offload=True | 推荐精度=bfloat16 / float16 | 原生FP8=False | 原生FP4=False


In [ ]:
# === 2. 挂载 20TB/30TB Google Drive（支持 colab-dfs-ext 零浏览器静默挂载）并初始化永久模型库目录 ===
import os
from pathlib import Path

MOUNT_POINT = "/content/drive"
if os.path.ismount(MOUNT_POINT):
    print(f"✅ 检测到 20TB/30TB Google Drive 已通过 colab-dfs-ext 静默挂载 ({MOUNT_POINT})，零弹窗直接复用！")
else:
    print(f"ℹ️ 尚未挂载 {MOUNT_POINT}（推荐本地执行 `colab dfs mount` 零浏览器挂载），正在回退至浏览器授权挂载...")
    from google.colab import drive
    drive.mount(MOUNT_POINT)

WORKSTATION_ROOT = Path(MOUNT_POINT) / "MyDrive" / "AI_Workstation"
DIRS = {
    "models": WORKSTATION_ROOT / "models",
    "loras": WORKSTATION_ROOT / "loras",
    "hf_cache": WORKSTATION_ROOT / "hf_cache",
    "outputs": WORKSTATION_ROOT / "outputs",
}

for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"📁 [{name}] 就绪: {path}")

# 将 HuggingFace 缓存永久绑定到 20TB/30TB 云盘，避免重复下载大模型
os.environ["HF_HOME"] = str(DIRS["hf_cache"])
os.environ["HUGGINGFACE_HUB_CACHE"] = str(DIRS["hf_cache"] / "hub")
print(f"\n✅ HF_HOME 已重定向至云盘: {os.environ['HF_HOME']}")

In [ ]:
# === 3. 工具箱：带「防烧钱保护」的 HuggingFace 模型云盘固化器 ===

def download_hf_model_to_drive(repo_id: str, local_dir_name: str = None, force_on_expensive_gpu: bool = False):
    """将 HuggingFace 模型直接下载固化到 20TB/30TB Google Drive 的 models 目录（自带高价 GPU 拦截保护）"""
    hw = inspect_hardware("download")
    target_dir = DIRS["models"] / (local_dir_name or repo_id.split("/")[-1])
    if target_dir.exists() and any(target_dir.iterdir()):
        print(f"✅ 模型已存在于 20TB/30TB 云盘，秒级复用无需下载: {target_dir}")
        return str(target_dir)

    if hw["ccu_per_hour"] >= 5.0 and not force_on_expensive_gpu:
        raise RuntimeError(
            f"🛑 已触发防烧钱拦截：当前挂载的是 [{hw['tier']}] ({hw['ccu_per_hour']} CCU/h)，"
            f"首次下载大模型建议使用 CPU (0 CCU) 或显式传入 force_on_expensive_gpu=True。"
        )

    from huggingface_hub import snapshot_download
    print(f"⬇️ 正在通过 Google 机房内网高速下载 {repo_id} -> {target_dir} ...")
    path = snapshot_download(repo_id=repo_id, local_dir=str(target_dir))
    print(f"🎉 下载完成，已永久保存至 20TB/30TB 云盘: {path}")
    return path

# 示例：下载 Wan2.1-T2V-1.3B 到 20TB/30TB 云盘（仅首次需下载）
# model_path = download_hf_model_to_drive("Wan-AI/Wan2.1-T2V-1.3B-Diffusers")

In [2]:
# === 4. 实战测试：运行 SnowGemma/Qwen2.5-1.5B-X-Writer-v1 ===
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1) 硬件智能评估 (1.5B 属于 llm_small，仅占 ~3.1GB)
hw = inspect_hardware("llm_small")

MODEL_ID = "SnowGemma/Qwen2.5-1.5B-X-Writer-v1"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16

print(f"\n🚀 正在加载模型 [{MODEL_ID}] 到 {device.upper()} ({dtype})...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map=device,
)
model.eval()
print(f"✅ 模型加载完成！耗时: {time.time() - t0:.1f}s")

# 2) 使用模型训练时的专用 System Prompt 与题目格式测试生成
SYSTEM_PROMPT = (
    "你是一名中文 X 内容写作者。根据用户提供的主题、素材和限制写作。"
    "保留具体事实和自然语气，不虚构经历、数据、引用或来源。"
    "长文使用自然段和必要的小标题，短推直接表达核心观点，回复简短口语化并直接回应对方。"
)

def write_x_post(user_prompt: str, max_new_tokens: int = 250):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(device)
    t_start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = outputs[0][inputs.input_ids.shape[1]:]
    cost_s = time.time() - t_start
    tps = len(gen_ids) / cost_s if cost_s > 0 else 0
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip(), len(gen_ids), cost_s, tps

# 测试 1：生成一条中文 X 短推
prompt_tweet = (
    "请写一条中文 X 短推。\n"
    "主题：在 Google Colab 上用 colab-mcp 和 G4 显卡跑大模型\n"
    "素材与要求：Google AI Ultra 5x 每月有 1000 CCU；Colab 里的 G4 是 96GB 显存的 RTX PRO 6000 Blackwell，一小时才 8.71 CCU；配合官方 colab-mcp 可以直接在本地 IDE 里让 Agent 远程操控云端显卡"
)
res1, tok1, sec1, tps1 = write_x_post(prompt_tweet)
print(f"\n📝 【测试 1：中文 X 短推】({tok1} tokens, {sec1:.2f}s, {tps1:.1f} tok/s)\n{'-'*50}\n{res1}\n{'-'*50}")

# 测试 2：生成一条中文 X 回复
prompt_reply = (
    "请写一条中文 X 回复。\n"
    "主题：有人问 Colab 里跑视频大模型和大语言模型选 A100 还是 G4\n"
    "素材与要求：直接选 G4，96GB Blackwell 显存比 A100 大，原生支持 FP8 和 FP4，算力翻倍但每小时只比 80G A100 贵 1.2 个 CCU"
)
res2, tok2, sec2, tps2 = write_x_post(prompt_reply)
print(f"\n💬 【测试 2：中文 X 回复】({tok2} tokens, {sec2:.2f}s, {tps2:.1f} tok/s)\n{'-'*50}\n{res2}\n{'-'*50}")

🖥️ 当前运行时: [CPU] CPU | 显存: 0.0 GB | 架构: Host CPU
💰 算力费率  : 0.0 CCU/小时 (按 1,000 CCU 测算可跑: ♾️ 无限 (0 CCU))
⚙️ 自动调优  : Offload=True | 推荐精度=bfloat16 / float16 | 原生FP8=False | 原生FP4=False

🚀 正在加载模型 [SnowGemma/Qwen2.5-1.5B-X-Writer-v1] 到 CPU (torch.bfloat16)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


✅ 模型加载完成！耗时: 23.4s

📝 【测试 1：中文 X 短推】(72 tokens, 27.64s, 2.6 tok/s)
--------------------------------------------------
Google AI Ultra每季度能给我1000CCU Google Colab上的G4显卡是96GB显存的RTX PRO 6000 Blackwell, 我的一小时才8.71CCU 配合官方colab-mcp可以让我在本地IDE里让Agent远程操控云端显卡
--------------------------------------------------

💬 【测试 2：中文 X 回复】(58 tokens, 22.17s, 2.6 tok/s)
--------------------------------------------------
直接选G4了，96GB的Blackwell显存，比A100要大很多 原生支持FP8和FP4，算力翻倍 但是每小时只有80G A100贵1.2个CCU
--------------------------------------------------
